# NB04b Peptide Identification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB04b_Peptide_Identification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB04a_NB04b/NB04b_Peptide_Identification.ipynb)

1. Understanding peptide identification from MS2 spectra
2. Matching and scoring of Tandem MS peptide identification
    3. Step 1: Data acquisition and preprocessing
    4. Step 2: Preselecting the peptides of interest
    5. Step 3+4: Matching and Scoring

## Understanding peptide identification from MS2 spectra

Under low-energy fragmentation conditions, peptide fragment patterns are reproducible and, in general, predictable, which allows for an 
amino acid sequence identification according to a fragmentation expectation. Algorithms for peptide identification perform in principle 
three basic tasks:

**(i)** a raw data preprocessing step is applied to the MS/MS spectra to obtain clean peak information. The same signal filtering 
and background subtraction methods are used as discussed in the section of low-level processing. Peak detection, however, may be performed 
differently. Preprocessing of MS/MS spectra focuses on the extraction of the precise m/z of the peak rather than the accurate peak areas. 
The conversion of a peak profile into the corresponding m/z and intensity values reduces the complexity, its representation is termed centroiding. 
To extract the masses for identification in a simple and fast way, peak fitting approaches are used. These approaches take either the most intense 
point of the peak profile, fit a Lorentzian distribution to the profile, or use a quadratic fit. 

**(ii)** Spectrum information and possible amino acid sequences assignments are evaluated. 

**(iii)** The quality of the match between spectrum and possible sequences is scored.

Even though the three steps roughly describe the basic principle of algorithms used for peptide sequence identification, most implementations 
show differences in individual steps which can lead to major changes in the outcome. Therefore, it has been proven useful to utilize more than 
one algorithm for a robust and thorough identification. Due to their major difference in identification strategies and prerequisites, 
identification algorithms are normally classified into three categories: (i) *de novo* peptide sequencing, (ii) peptide sequence-tag (PST) 
searching, and (iii) uninterpreted sequence searching. However, in this notebook we focus on the explanation of (iii) uninterpreted sequence 
searching, the most frequently used methods.
## Matching and scoring of Tandem MS peptide identification

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinIdentification.png)

**Figure 5: Process of computational identification of peptides from their fragment spectra**

Previously we learned, that peptides fragment to create patterns characteristic of a specific amino acid sequence. These patterns are reproducible and, in general, 
predictable taking the applied fragmentation method into account. This can be used for computational identification of peptides from their fragment spectra. 
This process can be subdivided into 5 main steps: spectrum preprocessing, selection of possible sequences, generating theoretical spectra, matching and scoring 
(Figure 5). The first step is a preprocessing of the experimental spectra and is done to reduce noise. Secondly, all possible amino acid 
sequences are selected which match the particular precursor peptide mass. The possible peptides can but do not need to be restricted to a particular organism. 
A theoretical spectrum is predicted for each of these amino acid sequences. Matching and scoring is performed by comparing experimental spectra to their predicted 
corresponding theoretical spectra. The score function measures the closeness of fit between the experimental acquired and theoretical spectrum. There are many 
different score functions, which can be considered as the main reason of different identifications considering different identification algorithm. The most 
natural score function is the cross correlation score (xcorr) used by the commercially available search tool SEQUEST. One of the reasons the xcorr is so 
sensitive is because it involves a correction factor that assesses the background correlation for each acquired spectrum and the theoretically predicted 
spectrum from sequences within a database. To compute this correction factor, a measure of similarity is calculated at different offsets between a preprocessed 
mass spectrum and a theoretical spectrum. The final xcorr is then the correlation at zero offset minus the mean correlation from all the individual offsets.

Thus, the correlation function measures the coherence of two signals by, in effect, translating one signal across the other. This can be mathematically 
achieved using the formula for cross-correlation in the form for discrete input signals:

***Equation 5***

![](https://latex.codecogs.com/png.latex?\large&space;R_{\tau}&space;=&space;\sum_{i=0}^{n-1}x_{i}\cdot%20y_{i&plus;\tau})

where x(i) is a peak of the reconstructed spectrum at position i and y(i) is a peak of the experimental spectrum. The displacement value 𝜏 
is the amount by which the signal is offset during the translation and is varied over a range of values. If two signals are the same, the correlation 
function should have its maxima at 𝜏=0, where there is no offset. The average of the offset-correlation is seen as the average background correlation 
and needs to be subtracted from the correlation at 𝜏=0. It follows: 

***Equation 6***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;R_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}R_{\tau})}{2*offset&plus;1})

In practice many theoretical spectra have to be matched again a single experimental spectrum. Therefore, the calculation can be speed up by reformulating Equation 5 and Equation 6 and introduce a preprocessing step, which is independent of the predicted spectra.

***Equation 7***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

For the preprocessed experimental spectrum y' it follows:

***Equation 8***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y`)

where:

***Equation 9***

![](https://latex.codecogs.com/png.latex?y'&space;=&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

Matching a measured spectrum against chlamy database


In [1]:
#r "nuget: BioFSharp, 2.0.0-beta4"
#r "nuget: BioFSharp.IO, 2.0.0-beta4"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.7"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: MzIO.SQL, 0.1.9"


open Plotly.NET
open BioFSharp
open BioFSharp.Mz
open System.IO
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL



Installed Packages BioFSharp, 2.0.0-beta4 BioFSharp.IO, 2.0.0-beta4 BioFSharp.Mz, 0.1.5-beta MzIO, 0.1.7 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.9 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

### Step 1: Data acquisition and preprocessing

We load  MS2 spectra that are saved in one mzlite file. The first step is to get the data out of the file


In [ ]:
//record type
type ms =  
    {
        mzIntensity: float array * float array
        retentiontime: float
        precursorMZ: float 
        id: string
    }


In [3]:
// Code-Block 1

//read in mzmlite
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"/home/paulinehans/Downloads/qPgr1_1_5 1.mzlite"|]

//set up connection between file and SQL database
let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let peaksReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let cn2 = peaksReader.Open()
let transaction = mzReader.BeginTransaction()



//function that returns all nessecary information we need, in this case m/z-ratio and intensity
let getMs2 = mzReader.ReadMassSpectra runID
let extractData =
    getMs2
    |> Seq.choose (fun x ->
        match MzIO.Processing.MassSpectrum.getMsLevel x with
        | 2 ->
            Some {
                id = x.ID
                precursorMZ = MzIO.Processing.MassSpectrum.getPrecursorMZ x
                retentiontime = MzIO.Processing.MassSpectrum.getScanTime x
                mzIntensity = 
                    PeakArray.mzIntensityArrayOf (
                        peaksReader.ReadSpectrumPeaks x.ID
                    )
            }
        | _ -> None
    )

    |> Seq.toArray

let allInfo = 
    extractData 
    |> Array.map (fun x -> x.id, x.precursorMZ, x.retentiontime, x.mzIntensity)
allInfo

index value 0 (sample=1 period=1 cycle=2174 experiment=2, 417.264562825677, 19.931816666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 417.264562825677 Item3 19.931816666667 Item4 (System.Double[], System.Double[]) Item1 [ 105.03694403378165, 115.08013500659963, 120.08544920767768, 129.05583050495537, 129.10590788474627, 130.05703540122514, 130.0823435909189, 130.09234814443013, 141.10754044611753, 147.11334269803564, 147.1248878264963, 171.18482641420263, 171.19226874292832, 173.10051455136585, 182.00514043510654, 185.15970377467337, 192.9860690277129, 198.56992893389665, 199.18500815998036, 200.1869457422182 ... (152 more) ] Item2 [ 0.28788364075467143, 1.5066630345160092, 6.925866867500474, 2.3933327269304527, 3.7502089226478574, 0.6406825098849822, 0.6407456320403071, 2.0024181259785223, 1.6683691980077242, 3.6625330068190465, 1.8739305793914696, 0.7350343651239655, 2.297050197988426, 0.7391373057998862, 1.1368653480019475, 1.1466768994466747, 0.7804389269949752, 1.187473543308215, 175.42347410108619, 9.140946516421877 ... (152 more) ] 1 (sample=1 period=1 cycle=2160 experiment=2, 599.844971734317, 19.84665, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2160 experiment=2 Item2 599.844971734317 Item3 19.84665 Item4 (System.Double[], System.Double[]) Item1 [ 103.0543497194824, 111.04821830863008, 120.0823717954965, 121.08322281956443, 175.12136387720136, 185.17020850371833, 186.15226148347944, 186.1740349161833, 199.1025260313089, 199.118140063875, 200.1128936655295, 213.16400115562234, 214.16525841575617, 217.13747208684208, 219.1582048950225, 227.09185579505618, 227.1056902398815, 227.94101973721743, 228.09742950931135, 228.11204228116964 ... (139 more) ] Item2 [ 0.8554617724200853, 1.4800350167348597, 22.162606387037442, 2.472752143083312, 2.694998218609612, 60.20195387502838, 1.149743547688331, 3.8327132281125387, 0.7927052338004614, 3.5673091662573597, 1.1920788413931405, 76.69131007070575, 4.727345968582597, 8.278353216739731, 2.495025970080178, 3.703830916839479, 9.312913757935348, 2.6505742508433627, 0.8484697828976095, 2.121253360037656 ... (139 more) ] 2 (sample=1 period=1 cycle=2158 experiment=2, 417.26171867267, 19.82215, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2158 experiment=2 Item2 417.26171867267 Item3 19.82215 Item4 (System.Double[], System.Double[]) Item1 [ 103.05722729347973, 120.08275436687575, 129.1076376798735, 130.08787586522436, 135.0666354525336, 147.11514036466176, 147.1269199829416, 157.10098836626625, 157.14110237989817, 167.07925479488196, 171.11542220841068, 171.19158778357942, 183.12217351526402, 198.57062385040575, 199.18460074875668, 200.18972708651606, 201.09064691675, 218.8583053346174, 218.86706174052853, 218.87581833242461 ... (197 more) ] Item2 [ 1.4257897058797653, 6.54097850474713, 4.308770141044249, 2.5630185710028286, 0.6529054293991976, 9.369269188661633, 0.5962553747792754, 1.0562262509154152, 1.408487089480559, 1.8154216598418316, 1.1023311371673117, 2.2051465403649217, 1.5204659618461847, 1.5832962719116495, 218.04026681015785, 16.19545596587386, 1.9916609557640186, 0.41555273010089877, 1.2466877788824604, 0.4155724558188467 ... (197 more) ] 3 (sample=1 period=1 cycle=2144 experiment=2, 599.849379512763, 19.737, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2144 experiment=2 Item2 599.849379512763 Item3 19.737 Item4 (System.Double[], System.Double[]) Item1 [ 103.05780600869107, 112.09632613764121, 119.71378299118587, 120.08496644756774, 121.09047782566947, 127.11327494301055, 157.1792975873414, 158.09537216204336, 175.12234625871776, 183.15444134973757, 184.59557355589055, 185.1668725980643, 186.17155238493572, 199.11130694360153, 200.11985045046197, 201.1403933804038, 211.1476224055335, 213.16390604700325, 214.1663291510673, 217.13695695154667 ... (127 more) ] Item2 [ 1.069349422593291, 1.1896025131363785, 0.3073394215221015, 33.01337073425472, 2.2409664290037767, 0.

Here, the spectrum is already centroidized as shown in *NB03c\_Centroidisation.ipynb* using the function 
`msPeakPicking`. So we just visualize mass and intensity:


In [4]:
// Code-Block 2

//first we only spectrum as example
let (ID,precursorMz, rt, (mz,intensity)) = 
    allInfo 
    |> Array.find (fun (ID, precursorMz, rt, (mz, intensity)) -> 
        ID.Contains("sample=1 period=1 cycle=1574 experiment=5"))

let test = 
    allInfo 
    |> Array.find (fun (ID, precursorMz, rt, (mz, intensity)) -> 
        ID.Contains("sample=1 period=1 cycle=1574 experiment=5"))
//creating chart, which visualizes m/z & intensity
let ms2Chart =  
    let chart = 
        Chart.Column(values = intensity, Keys = mz, Width = 8.)
        |> Chart.withYAxisStyle("Intensity")
        |> Chart.withXAxisStyle("m/z")
    chart
ms2Chart
test

(sample=1 period=1 cycle=1574 experiment=5, 909.46774146616, 9.350283333333, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=1574 experiment=5 Item2 909.46774146616 Item3 9.350283333333 Item4 (System.Double[], System.Double[]) Item1 [ 129.10731264486066, 130.09284972719183, 141.1026531292033, 143.12161112275044, 147.11720964562807, 157.0623119517569, 157.13558460229936, 159.08061824082387, 160.08660666164312, 169.14707774733228, 171.11585548096912, 172.12229441600468, 185.12975758144805, 186.12341164116583, 186.13842134388042, 199.0861592622142, 199.1834723940368, 203.07427889328633, 204.1393424178031, 205.1309416478466 ... (361 more) ] Item2 [ 11.489966263297902, 2.8834755805632994, 1.0010024874361534, 93.33685220596237, 9.198945463265318, 1.0560963940463353, 30.810419700017633, 8.503001073097494, 1.0662156830562708, 2.1919458041963935, 92.22848076314563, 7.738964520066816, 48.3475218105923, 5.173476083550099, 1.1497027759219236, 4.756068404799407, 8.325211619904849, 9.606820215179823, 19.866182511427155, 1.2069329768394255 ... (361 more) ]

In [ ]:
// Code-Block 3

let lowerScanLimit = 150.
let upperScanLimit = 1000.

//all we need
let makingDataPretty = 
    allInfo 
    |> Array.map (fun (ID, precursorMz, rt, (mz, intensity)) -> ID, precursorMz, mz, intensity)

//normalization
let preprocessedIntesities = 
    makingDataPretty 
    |> Array.map (fun (ID,precursorMz, mz,int) -> 
        let normalization =
            Mz.PeakArray.zip mz int
            |> (fun pa -> Mz.PeakArray.peaksToNearestUnitDaltonBinVector pa lowerScanLimit upperScanLimit)
            |> (fun pa -> Mz.SequestLike.windowNormalizeIntensities pa 10)
        ID,precursorMz,normalization
    )
preprocessedIntesities


//example Spectrum Chart
let exampleData = 
    preprocessedIntesities
    |> Array.find (fun (ID, precursorMz, vector) -> 
        ID.Contains("sample=1 period=1 cycle=1574 experiment=5"))

let (id, precursorMz,vec) = exampleData
let keysValues =
    vec
    |> Seq.mapi (fun i intensity ->
        let mz = lowerScanLimit + float i   
        (mz, intensity)
    )
    |>Seq.toArray
    
let exampleChart = 
        Chart.Column (keysValues = keysValues , Width = 1. )
        |> Chart.withSize(1400,800)
    
exampleChart

<!-- Plotly chart will be drawn inside this DIV -->

Now, we will preprocess the experimental spectrum from our example. This is on the one hand to reduce noise, but also to make 
the measured spectrum more like the once we are able to simulate. 


### Step 2: Preselecting the peptides of interest
From our previos notebook *NB02b\_Digestion\_and\_mass\_calculation.ipynb*, we know how to 
calculate all peptide masses that we can expect to be present in *Chlamydomonas reinhardtii*.




In [6]:
// Code-Block 4
let path2 = Path.Combine[|directory;"downloads/Chlamy_JGI5_5_QProt.fasta"|]

let peptideAndMasses = 
    path2
    |> IO.FastA.fromFile BioArray.ofAminoAcidString
    |> Seq.toArray
    |> Array.mapi (fun i fastAItem ->
        Digestion.BioArray.digest Digestion.Table.Trypsin i fastAItem.Sequence
        |> Digestion.BioArray.concernMissCleavages 0 0
    )
    |> Array.concat
    |> Array.map (fun peptide ->
        // calculate mass for each peptide
        peptide.PepSequence, BioSeq.toMonoisotopicMassWith (BioItem.monoisoMass ModificationInfo.Table.H2O) peptide.PepSequence
    )

peptideAndMasses |> Array.head


([Leu; Ala; Ala; ... ], 1408.6860982223602) Item1 [ Leu, Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His Head Ala Tail [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His (values) index value 0 Ala 1 Ala 2 Ala 3 Leu 4 Glu 5 His 6 His 7 His 8 His 9 His 10 His Head Leu Tail [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 Hi

In the previous notebook *NB04a\_Fragmentation\_for\_peptide\_identification.ipynb*, 
we used functions that generate the theoretical series of b- and y-ions from the given peptide. Combined with the function 
`Mz.SequestLike.predictOf` that generates theoretical spectra that fit the Sequest scoring algorithm.


In [7]:
// Code-Block 6

let predictFromSequence peptide =
    [
        peptide
        |> Mz.Fragmentation.Series. yOfBioList BioItem.initMonoisoMassWithMemP
        peptide
        |> Mz.Fragmentation.Series.bOfBioList BioItem.initMonoisoMassWithMemP
    ]
    |> List.concat
    |> Mz.SequestLike.predictOf (lowerScanLimit,upperScanLimit) 2.
 

### Step 3+4: Matching and Scoring

In the matching step, we compare theoretical spectra of peptides within our precursor peptide mass range with our measured spectra. 
We get a score which tells us how well the theoretical spectrum fits the given experimental spectrum.


In [8]:
// Code-Block 7

let sortedScores = 
    let spectras = 
        preprocessedIntesities 
        |> Array.map (fun x -> 
            let calc = 
                peptideAndMasses
                |> Array.map (fun (sequence,mass)    ->
                    sequence,predictFromSequence sequence
                )
                |> Array.map (fun (sequence,theoSpectrum) -> 
                    sequence,BioFSharp.Mz.SequestLike.scoreSingle theoSpectrum (snd x))
                
                |> Array.sortByDescending (fun (sequence,score) -> score)
            fst x, calc
        )
    spectras
sortedScores 

index value 0 (sample=1 period=1 cycle=2174 experiment=2, System.Tuple`2[Microsoft.FSharp.Collections.FSharpList`1[BioFSharp.AminoAcids+AminoAcid],System.Double][]) Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 index value 0 ([Leu; Leu; Phe; ... ], 7.463680414533675) Item1 [ Leu, Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] (values) index value 0 Leu 1 Phe 2 Glu 3 Ala 4 Leu 5 Lys Head Leu Tail [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] (values) index value 0 Leu 1 Phe 2 Glu 3 Ala 4 Leu 5 Lys (values) index value 0 Leu 1 Leu 2 Phe 3 Glu 4 Ala 5 Leu 6 Lys Item2 7.463680414533675 1 ([Ala; Leu; Gln; ... ], 2.789012136050165) Item1 [ Ala, Leu, Gln, Asn, Thr, Val, Leu, Lys ] HeadOrDefault Ala TailOrNull [ Leu, Gln, Asn, Thr, Val, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Gln, Asn, Thr, Val, Leu, Lys ] Head Leu Tail [ Gln, Asn, Thr, Val, Leu, Lys ] (values) index value 0 Leu 1 Gln 2 Asn 3 Thr 4 Val 5 Leu 6 Lys Head Ala Tail [ Leu, Gln, Asn, Thr, Val, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Gln, Asn, Thr, Val, Leu, Lys ] Head Leu Tail [ Gln, Asn, Thr, Val, Leu, Lys ] (values) index value 0 Leu 1 Gln 2 Asn 3 Thr 4 Val 5 Leu 6 Lys (values) index value 0 Ala 1 Leu 2 Gln 3 Asn 4 Thr 5 Val 6 Leu 7 Lys Item2 2.789012136050165 2 ([Leu; Ser; Ile; ... ], 2.1351361285596027) Item1 [ Leu, Ser, Ile, Phe, Glu, Thr, Gly, Ile, Lys ] HeadOrDefault Leu TailOrNull [ Ser, Ile, Phe, Glu, Thr, Gly, Ile, Lys ] HeadOrDefault Ser TailOrNull [ Ile, Phe, Glu, Thr, Gly, Ile, Lys ] Head Ser Tail [ Ile, Phe, Glu, Thr, Gly, Ile, Lys ] (values) index value 0 Ser 1 Ile 2 Phe 3 Glu 4 Thr 5 Gly 6 Ile 7 Lys Head Leu Tail [ Ser, Ile, Phe, Glu, Thr, Gly, Ile, Lys ] HeadOrDefault Ser TailOrNull [ Ile, Phe, Glu, Thr, Gly, Ile, Lys ] Head Ser Tail [ Ile, Phe, Glu, Thr, Gly, Ile, Lys ] (values) index value 0 Ser 1 Ile 2 Phe 3 Glu 4 Thr 5 Gly 6 Ile 7 Lys (values) index value 0 Leu 1 Ser 2 Ile 3 Phe 4 Glu 5 Thr 6 Gly 7 Ile 8 Lys Item2 2.1351361285596027 3 ([Val; Thr; Glu; ... ], 2.100388603200556) Item1 [ Val, Thr, Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] HeadOrDefault Val TailOrNull [ Thr, Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] HeadOrDefault Thr TailOrNull [ Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] Head Thr Tail [ Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] (values) index value 0 Thr 1 Glu 2 Ala 3 Ala 4 Ala 5 Leu 6 Ala 7 Ser 8 Gly 9 Arg Head Val Tail [ Thr, Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] HeadOrDefault Thr TailOrNull [ Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] Head Thr Tail [ Glu, Ala, Ala, Ala, Leu, Ala, Ser, Gly, Arg ] (values) index value 0 Thr 1 Glu 2 Ala 3 Ala 4 Ala 5 Leu 6 Ala 7 Ser 8 Gly 9 Arg (values) index value 0 Val 1 Thr 2 Glu 3 Ala 4 Ala 5 Ala 6 Leu 7 Ala 8 Ser 9 Gly 10 Arg Item2 2.100388603200556 4 ([Ile; Tyr; Ser; ... ], 1.5828601468445378) Item1 [ Ile, Tyr, Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] HeadOrDefault Ile TailOrNull [ Tyr, Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] HeadOrDefault Tyr TailOrNull [ Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] Head Tyr Tail [ Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] (values) index value 0 Tyr 1 Ser 2 Phe 3 Asn 4 Glu 5 Gly 6 Asn 7 Tyr 8 Gly 9 Leu 10 Trp 11 Asp 12 Asp 13 Ser 14 Val 15 Lys Head Ile Tail [ Tyr, Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] HeadOrDefault Tyr TailOrNull [ Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] Head Tyr Tail [ Ser, Phe, Asn, Glu, Gly, Asn, Tyr, Gly, Leu, Trp, Asp, Asp, Ser, Val, Lys ] (values) index value 0 Tyr 1 Ser 2 Phe 3 Asn 4 Glu 5 Gly 6 Asn 7 Tyr 8 Gly 9 Leu 10 Trp 11 Asp 12 Asp 13 Ser 14 Val 15 Lys (values) index

So short recap: we compared all experimental spectra in the mzlite file with theoretical spectra from the FASTA file. Now we take the best matching scores. This will tell us, the most probable peptide sequence.

In [9]:
// Code-Block 8
let bestScore = 
    sortedScores 
    |> Array.map (fun (x,y) ->x, y |> Array.head)
bestScore



index value 0 (sample=1 period=1 cycle=2174 experiment=2, ([Leu; Leu; Phe; ... ], 7.463680414533675)) Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 ([Leu; Leu; Phe; ... ], 7.463680414533675) Item1 [ Leu, Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys (values) index value 0 Leu 1 Phe 2 Glu 3 Ala 4 Leu 5 Lys Head Leu Tail [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys (values) index value 0 Leu 1 Phe 2 Glu 3 Ala 4 Leu 5 Lys (values) index value 0 Leu 1 Leu 2 Phe 3 Glu 4 Ala 5 Leu 6 Lys Item2 7.463680414533675 1 (sample=1 period=1 cycle=2160 experiment=2, ([Leu; Val; Phe; ... ], 9.395028227159235)) Item1 sample=1 period=1 cycle=2160 experiment=2 Item2 ([Leu; Val; Phe; ... ], 9.395028227159235) Item1 [ Leu, Val, Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Leu TailOrNull [ Val, Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Val TailOrNull [ Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Phe TailOrNull [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] Head Phe Tail [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] (values) index value 0 Phe 1 Pro 2 Glu 3 Glu 4 Val 5 Leu 6 Pro 7 Arg Head Val Tail [ Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Phe TailOrNull [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] Head Phe Tail [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] (values) index value 0 Phe 1 Pro 2 Glu 3 Glu 4 Val 5 Leu 6 Pro 7 Arg (values) index value 0 Val 1 Phe 2 Pro 3 Glu 4 Glu 5 Val 6 Leu 7 Pro 8 Arg Head Leu Tail [ Val, Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Val TailOrNull [ Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Phe TailOrNull [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] Head Phe Tail [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] (values) index value 0 Phe 1 Pro 2 Glu 3 Glu 4 Val 5 Leu 6 Pro 7 Arg Head Val Tail [ Phe, Pro, Glu, Glu, Val, Leu, Pro, Arg ] HeadOrDefault Phe TailOrNull [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] Head Phe Tail [ Pro, Glu, Glu, Val, Leu, Pro, Arg ] (values) index value 0 Phe 1 Pro 2 Glu 3 Glu 4 Val 5 Leu 6 Pro 7 Arg (values) index value 0 Val 1 Phe 2 Pro 3 Glu 4 Glu 5 Val 6 Leu 7 Pro 8 Arg (values) index value 0 Leu 1 Val 2 Phe 3 Pro 4 Glu 5 Glu 6 Val 7 Leu 8 Pro 9 Arg Item2 9.395028227159235 2 (sample=1 period=1 cycle=2158 experiment=2, ([Leu; Leu; Phe; ... ], 7.302567421626155)) Item1 sample=1 period=1 cycle=2158 experiment=2 Item2 ([Leu; Leu; Phe; ... ], 7.302567421626155) Item1 [ Leu, Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys (values) index value 0 Leu 1 Phe 2 Glu 3 Ala 4 Leu 5 Lys Head Leu Tail [ Leu, Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Leu TailOrNull [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOrNull [ Glu, Ala, Leu, Lys ] Head Phe Tail [ Glu, Ala, Leu, Lys ] (values) index value 0 Phe 1 Glu 2 Ala 3 Leu 4 Lys Head Leu Tail [ Phe, Glu, Ala, Leu, Lys ] HeadOrDefault Phe TailOr

Simply lovely its done but notice however, that in a real world we would need to 
relate our score to the complete data set to get an idea of the overall quality and which numerical value we could trust. 

## Questions

1. How does the Chart change, when you change the value of 'numberofwindows' from 10 to 20?
  What does this parameter specify? (Code-Block 3)
2. What is the rational behind the normalization of measured spectra?
3. Why are we adding the mass of water to the peptide sequence? (BioItem.monoisoMass ModificationInfo.Table.H2O) (Code-Block 4)
4. In code block 6 you get a raw estimate on how many peptide sequences are candidates for a match, when given a certain mz.
Given that one MS run can consist of up to 120.000 ms2 spectra, how many peptide spectrum matches do you have to perform?
What does that mean for the performance of the application? Which parameters influence the size of the "search space"? (Code-Block 6)
5. What happens when you choose different lower and upper scan limits?
6. Visualize the distribution of scores using a histogram. (Code-Block 7)
